# Описание задачи

Компания **MTS Web Services** планирует улучшить свою рекомендательную систему для видеохостинга, чтобы предложить пользователям тот контент, который они с большой вероятностью посмотрят до конца (или хотя бы до половины).

---

## Задание

Вашей задачей является разработка модели машинного обучения, способной прогнозировать вероятность просмотра видеоролика пользователем более чем на 50%.

На основе имеющихся данных вам предстоит реализовать следующий план действий:

- Реализуйте классификационную модель, позволяющую определять, будет ли видеоролик просмотрен пользователем больше чем на половину.
- Получите прогнозы на тестовых данных, содержащихся в файле `interactions_public_test.csv`.
- Подготовьте и загрузите CSV-файл с вашими прогнозами.

📌 Обратите особое внимание на следующее правило: категорически запрещается использование данных из открытых (`public_test`) и закрытых (`private_test`) тестовых выборок для дополнительного обучения или тонкой настройки вашей модели.

Также важно обеспечить воспроизводимость полученных вами результатов. Обязательно зафиксируйте начальное значение генератора случайных чисел (**seed**) перед началом обучения вашего алгоритма. Выбирая seed, делайте это случайно каждый раз и обязательно включайте выбранное значение в имя итогового файла с результатами.

---

## Данные

Исходный набор данных предварительно разбит на три части:

- Обучающая выборка
- Открытая тестовая выборка
- Закрытая тестовая выборка

Размеры каждой из частей примерно такие:

| Тип выборки      | Количество записей |
|------------------|:-----------------:|
| Обучающая        |       754 313     |
| Тестовая открытая|   ~100 577–100 782 |
| Тестовая закрытая|   ~100 577–100 782 |

### Предоставленные файлы:

- `interactions_train.csv`: обучающие данные.
- `users.csv`: метаданные пользователей.
- `items.csv`: метаданные видеоконтента.
- `interactions_public_test.csv`: тестовые данные.
- `sample_sub_public_test_seed_0.csv`: пример формата подачи прогнозов.

---

## Структура входных данных

### Поля в обучающих данных (`interactions_train.csv`)

- `user_id`: Уникальный идентификатор пользователя (по данному полю возможно получение дополнительной информации о пользователе из файла `users.csv`).
- `item_id`: Уникальный идентификатор видео (по данному полю возможно получение дополнительной информации о видео из файла `items.csv`).
- `last_watch_dt`: Дата и время последнего просмотра видео.
- `total_dur`: Общая длительность видео.
- `watched_pct`: Процент просмотра видео пользователем.

### Метаданные пользователей (`users.csv`)

- `user_id`: Идентификатор пользователя.
- `age`: Возрастная категория пользователя.
- `income`: Уровень дохода.
- `sex`: Пол.
- `kids_flg`: Флаг детского аккаунта.

### Метаданные видеоконтента (`items.csv`)

- `item_id`: Идентификатор видео.
- `content_type`: Тип контента.
- `title`: Заголовок видео.
- `title_orig`: Оригинальное название/идентификационная ссылка.
- `release_year`: Год выхода видео.
- `genres`: Жанр(-ы) видео.
- `countries`: Страна(-ы) производства.
- `for_kids`: Детский контент.
- `age_rating`: Возрастной рейтинг.
- `studios`: Студии-производители.
- `directors`: Режиссёры.
- `actors`: Актеры.
- `keywords`: Ключевые слова.

---

## Оценочная метрика

Качество предложенной модели оценивается по показателю **F1 макроусреднения (macro)**.

---

## Формат вывода

Итоговый файл должен представлять собой таблицу в формате CSV с пятью столбцами, аналогично структуре файла `interactions_train.csv`. Первая строка файла обязана содержать имена соответствующих столбцов (пример см. в образце файла `public_sample_submission_seed_0.csv`). Все последующие строки должны включать ваши рассчитанные значения.

Прогнозируемые значения должны находиться в пределах диапазонов `[0, 100]` или `[0, 1]`. Граница принятия решения устанавливается равной 50% или 0.5 соответственно, в зависимости от выбранного вами диапазона.

Имя итогового файла должно обязательно включать использованный вами seed, например: `submission_seed_<значение>.csv`.


In [25]:
# ⚙️ Настройка окружения и подключение необходимых модулей
import pandas as pd
import numpy as np

In [26]:
# Загружаем train_df из CSV
train_df = pd.read_csv('interactions_train.csv')

# Загружаем users_df из CSV
users_df = pd.read_csv('users.csv')

# Загружаем items_df из CSV
items_df = pd.read_csv('items.csv')

In [27]:
# Загрузка публичной тестовой выборки
test_df = pd.read_csv("interactions_public_test.csv")

In [28]:
print(test_df.head().to_markdown())
print(test_df.info())

|    |   user_id |   item_id | last_watch_dt   |   total_dur |
|---:|----------:|----------:|:----------------|------------:|
|  0 |         1 |      5938 | 2021-06-20      |       23721 |
|  1 |         2 |     11312 | 2021-08-14      |        8442 |
|  2 |         3 |      9353 | 2021-08-21      |          28 |
|  3 |         9 |     15391 | 2021-08-19      |        7331 |
|  4 |        12 |      7727 | 2021-08-04      |        7827 |
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100577 entries, 0 to 100576
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   user_id        100577 non-null  int64  
 1   item_id        100577 non-null  int64  
 2   last_watch_dt  100577 non-null  object 
 3   total_dur      100577 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 3.1+ MB
None


In [29]:
sub = pd.read_csv("sample_sub_public_test_seed_0.csv")

In [30]:
print(sub.head().to_markdown())
print(sub.info())

|    |   user_id |   item_id | last_watch_dt   |   total_dur |   watched_pct |
|---:|----------:|----------:|:----------------|------------:|--------------:|
|  0 |         1 |      5938 | 2021-06-20      |       23721 |             0 |
|  1 |         2 |     11312 | 2021-08-14      |        8442 |             0 |
|  2 |         3 |      9353 | 2021-08-21      |          28 |             0 |
|  3 |         9 |     15391 | 2021-08-19      |        7331 |             0 |
|  4 |        12 |      7727 | 2021-08-04      |        7827 |             0 |
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100577 entries, 0 to 100576
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   user_id        100577 non-null  int64  
 1   item_id        100577 non-null  int64  
 2   last_watch_dt  100577 non-null  object 
 3   total_dur      100577 non-null  float64
 4   watched_pct    100577 non-null  float64
dtypes: float64(2), in

In [31]:
print(train_df.head().to_markdown())
print(train_df.info())

|    |   user_id |   item_id | last_watch_dt   |   total_dur |   watched_pct |
|---:|----------:|----------:|:----------------|------------:|--------------:|
|  0 |         3 |     10139 | 2021-04-25      |         103 |             2 |
|  1 |         3 |      7204 | 2021-07-17      |          28 |             1 |
|  2 |         3 |     12928 | 2021-08-17      |         845 |            15 |
|  3 |         3 |      3897 | 2021-08-17      |         901 |            15 |
|  4 |         4 |       811 | 2021-06-12      |        6191 |           100 |
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 922967 entries, 0 to 922966
Data columns (total 5 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   user_id        922967 non-null  int64  
 1   item_id        922967 non-null  int64  
 2   last_watch_dt  922967 non-null  object 
 3   total_dur      922966 non-null  float64
 4   watched_pct    922782 non-null  float64
dtypes: float64(2), in

In [32]:
print(items_df.head(3).to_markdown())
print(items_df.info())

|    |   item_id | content_type   | title       | title_orig       |   release_year | genres                                                               | countries      |   for_kids |   age_rating |   studios | directors       | actors                                                                                                                                                                                                                                                                                 | keywords                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
|---:|----------:|:---------------|:--

In [33]:
# Основные статистики для watched_pct
print("Статистики по watched_pct в train_df:")
print(train_df['watched_pct'].describe())

# Посмотрим, сколько значений > 50 и <= 50
# Предположим, что watched_pct в диапазоне [0, 100]
high_watched_count = (train_df['watched_pct'] > 50).sum()
low_watched_count = (train_df['watched_pct'] <= 50).sum()
print(f"\nКоличество просмотров > 50%: {high_watched_count}")
print(f"Количество просмотров <= 50%: {low_watched_count}")

# Рассчитаем процент
total_count = len(train_df)
high_watched_pct = (high_watched_count / total_count) * 100
low_watched_pct = (low_watched_count / total_count) * 100
print(f"Процент просмотров > 50%: {high_watched_pct:.2f}%")
print(f"Процент просмотров <= 50%: {low_watched_pct:.2f}%")

# Проверим уникальные значения для понимания шкалы
print(f"\nМинимальное значение watched_pct: {train_df['watched_pct'].min()}")
print(f"Максимальное значение watched_pct: {train_df['watched_pct'].max()}")
print(f"Количество уникальных значений watched_pct: {train_df['watched_pct'].nunique()}")

Статистики по watched_pct в train_df:
count    922782.000000
mean         49.697284
std          42.560386
min           0.000000
25%           5.000000
50%          41.000000
75%         100.000000
max         100.000000
Name: watched_pct, dtype: float64

Количество просмотров > 50%: 430664
Количество просмотров <= 50%: 492118
Процент просмотров > 50%: 46.66%
Процент просмотров <= 50%: 53.32%

Минимальное значение watched_pct: 0.0
Максимальное значение watched_pct: 100.0
Количество уникальных значений watched_pct: 101


Классы почти сбалансированы, что хорошо для задачи бинарной классификации.

In [34]:
# Проверка пропусков
print("Пропуски в train_df:")
print(train_df.isnull().sum())
print("\nПропуски в users_df:")
print(users_df.isnull().sum())
print("\nПропуски в items_df:")
print(items_df.isnull().sum())

# Проверка дубликатов в train_df по user_id и item_id
duplicate_rows = train_df.duplicated(subset=['user_id', 'item_id']).sum()
print(f"\nКоличество дубликатов в train_df по (user_id, item_id): {duplicate_rows}")

Пропуски в train_df:
user_id            0
item_id            0
last_watch_dt      0
total_dur          1
watched_pct      185
dtype: int64

Пропуски в users_df:
user_id         0
age         14095
income      14776
sex         13831
kids_flg        0
dtype: int64

Пропуски в items_df:
item_id             0
content_type        0
title               0
title_orig       4745
release_year       98
genres              0
countries          37
for_kids        15397
age_rating          2
studios         14898
directors        1509
actors           2619
keywords          423
dtype: int64

Количество дубликатов в train_df по (user_id, item_id): 0


In [35]:
# Создаем бинарную целевую переменную
# Предполагаем, что watched_pct в диапазоне [0, 100], порог 50%
train_df['target'] = (train_df['watched_pct'] > 50).astype(int)

# Удаляем строки, где watched_pct — NaN
print(f"Количество строк в train_df до очистки: {len(train_df)}")
train_df_clean = train_df.dropna(subset=['watched_pct'])
print(f"Количество строк в train_df после удаления NaN в watched_pct: {len(train_df_clean)}")

# Сводка по новому датафрейму
print("\nРаспределение целевой переменной 'target' после очистки:")
print(train_df_clean['target'].value_counts())

Количество строк в train_df до очистки: 922967
Количество строк в train_df после удаления NaN в watched_pct: 922782

Распределение целевой переменной 'target' после очистки:
target
0    492118
1    430664
Name: count, dtype: int64


In [36]:
# Объединяем train_df_clean с users_df по user_id
train_full = train_df_clean.merge(users_df, on='user_id', how='left')

# Затем объединяем с items_df по item_id
train_full = train_full.merge(items_df, on='item_id', how='left')

print("Форма итогового датасета после объединения:")
print(train_full.shape)
print("\nИнформация о типах данных и пропусках в объединенном датасете:")
print(train_full.info())

Форма итогового датасета после объединения:
(922782, 22)

Информация о типах данных и пропусках в объединенном датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 922782 entries, 0 to 922781
Data columns (total 22 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   user_id        922782 non-null  int64  
 1   item_id        922782 non-null  int64  
 2   last_watch_dt  922782 non-null  object 
 3   total_dur      922782 non-null  float64
 4   watched_pct    922782 non-null  float64
 5   target         922782 non-null  int64  
 6   age            743987 non-null  object 
 7   income         744296 non-null  object 
 8   sex            743893 non-null  object 
 9   kids_flg       754313 non-null  float64
 10  content_type   922782 non-null  object 
 11  title          922782 non-null  object 
 12  title_orig     859035 non-null  object 
 13  release_year   922716 non-null  float64
 14  genres         922782 non-null  object 
 15

In [37]:
# Список столбцов для анализа
cat_features = ['content_type', 'age', 'income', 'sex', 'for_kids', 'kids_flg']

print("Уникальные значения и их количество для категориальных признаков:")
for col in cat_features:
    print(f"\n--- {col} ---")
    print(f"Количество уникальных значений: {train_full[col].nunique()}")
    print(f"Топ-5 значений:")
    print(train_full[col].value_counts(dropna=False).head())

Уникальные значения и их количество для категориальных признаков:

--- content_type ---
Количество уникальных значений: 2
Топ-5 значений:
content_type
film      731628
series    191154
Name: count, dtype: int64

--- age ---
Количество уникальных значений: 6
Топ-5 значений:
age
age_25_34    219138
age_35_44    214456
NaN          178795
age_45_54    129388
age_18_24     91083
Name: count, dtype: int64

--- income ---
Количество уникальных значений: 6
Топ-5 значений:
income
income_20_40    415269
income_40_60    232904
NaN             178486
income_60_90     62815
income_0_20      19524
Name: count, dtype: int64

--- sex ---
Количество уникальных значений: 2
Топ-5 значений:
sex
М      379832
Ж      364061
NaN    178889
Name: count, dtype: int64

--- for_kids ---
Количество уникальных значений: 2
Топ-5 значений:
for_kids
NaN    902709
0.0     18981
1.0      1092
Name: count, dtype: int64

--- kids_flg ---
Количество уникальных значений: 2
Топ-5 значений:
kids_flg
0.0    500366
1.0    2539

In [38]:
import pandas as pd

# Пример: средний target по content_type
print("Средний target по content_type:")
print(train_full.groupby('content_type')['target'].agg(['mean', 'count']))

# Средний target по kids_flg
print("\nСредний target по kids_flg:")
print(train_full.groupby('kids_flg')['target'].agg(['mean', 'count']))

# Средний target по sex
print("\nСредний target по sex:")
print(train_full.groupby('sex')['target'].agg(['mean', 'count']))

# Средний target по age
print("\nСредний target по age:")
print(train_full.groupby('age')['target'].agg(['mean', 'count']))

Средний target по content_type:
                  mean   count
content_type                  
film          0.515322  731628
series        0.280611  191154

Средний target по kids_flg:
              mean   count
kids_flg                  
0.0       0.456590  500366
1.0       0.484782  253947

Средний target по sex:
         mean   count
sex                  
Ж    0.469246  364061
М    0.464284  379832

Средний target по age:
                mean   count
age                         
age_18_24   0.470373   91083
age_25_34   0.475645  219138
age_35_44   0.480961  214456
age_45_54   0.455127  129388
age_55_64   0.436663   56918
age_65_inf  0.397861   33004


- **`content_type`**: Видео типа `film` имеют **средний `target` 0.515**, тогда как `series` — **всего 0.281**. Это **очень сильный признак**. Люди с гораздо большей вероятностью досматривают фильмы до 50% и более, 
- **`age`**: Есть **четкая тенденция** — с увеличением возраста средний `target` **уменьшается**. От `age_35_44` (0.481) до `age_65_inf` (0.398). Это говорит о том, что возраст — важный признак.


In [39]:
# Заполним пропуски в категориальных признаках значением 'Unknown'
cat_features_to_fill = ['age', 'income', 'sex']
for col in cat_features_to_fill:
    train_full[col] = train_full[col].fillna('Unknown')

# Заполним пропуски в числовых признаках (for_kids, release_year, kids_flg) значением -1
num_features_to_fill = ['for_kids', 'release_year', 'kids_flg']
for col in num_features_to_fill:
    train_full[col] = train_full[col].fillna(-1)

# Проверим, остались ли пропуски
print("Пропуски в train_full после заполнения:")
print(train_full[num_features_to_fill + cat_features_to_fill + ['genres', 'countries', 'studios', 'directors', 'actors', 'keywords']].isnull().sum())

Пропуски в train_full после заполнения:
for_kids             0
release_year         0
kids_flg             0
age                  0
income               0
sex                  0
genres               0
countries           39
studios         915064
directors         8592
actors           27297
keywords         37992
dtype: int64


LGBM может работать с категориальными признаками напрямую, если указать их при обучении, но для начала посмотрим, как они выглядят после заполнения пропусков.

In [40]:
# Посмотрим уникальные значения в заполненных категориальных признаках
cat_features_for_encoding = ['content_type', 'age', 'income', 'sex']

print("Уникальные значения в категориальных признаках после заполнения NaN:")
for col in cat_features_for_encoding:
    print(f"\n--- {col} ---")
    print(f"Количество уникальных значений: {train_full[col].nunique()}")
    print(f"Значения: {list(train_full[col].unique())}")

Уникальные значения в категориальных признаках после заполнения NaN:

--- content_type ---
Количество уникальных значений: 2
Значения: ['film', 'series']

--- age ---
Количество уникальных значений: 7
Значения: ['age_35_44', 'Unknown', 'age_25_34', 'age_55_64', 'age_65_inf', 'age_45_54', 'age_18_24']

--- income ---
Количество уникальных значений: 7
Значения: ['income_20_40', 'Unknown', 'income_40_60', 'income_90_150', 'income_60_90', 'income_0_20', 'income_150_inf']

--- sex ---
Количество уникальных значений: 3
Значения: ['Ж', 'Unknown', 'М']


Мы пока **не включаем** сложные признаки вроде `genres`, `countries`, `studios`, `directors`, `actors`, `keywords`, так как они требуют более сложной обработки. Начнем с простых и посмотрим, как модель себя покажет.

Для кодирования будем использовать **Label Encoding**, так как `LGBM` неплохо справляется с такими признаками, особенно если указать их как `categorical_features`.

In [41]:
from sklearn.preprocessing import LabelEncoder

# Список признаков для модели
features_to_use = ['content_type', 'age', 'income', 'sex', 'kids_flg', 'total_dur', 'release_year']

# Проверим, все ли колонки присутствуют в train_full
print("Колонки в train_full:", list(train_full.columns))
print("\nВыбранные признаки:", features_to_use)
print("Отсутствующие признаки:", [col for col in features_to_use if col not in train_full.columns])

# Применим Label Encoding к категориальным признакам
categorical_features = ['content_type', 'age', 'income', 'sex']

le_dict = {}
for col in categorical_features:
    le = LabelEncoder()
    # Fit на уникальных значениях из train_full для надежности
    le.fit(train_full[col].astype(str))
    train_full[col + '_le'] = le.transform(train_full[col].astype(str))
    le_dict[col] = le  # Сохраним для последующего использования на тесте

# Теперь используем закодированные признаки и числовые
final_features = [col + '_le' if col in categorical_features else col for col in features_to_use]

print("\nФинальные признаки для модели:", final_features)

Колонки в train_full: ['user_id', 'item_id', 'last_watch_dt', 'total_dur', 'watched_pct', 'target', 'age', 'income', 'sex', 'kids_flg', 'content_type', 'title', 'title_orig', 'release_year', 'genres', 'countries', 'for_kids', 'age_rating', 'studios', 'directors', 'actors', 'keywords']

Выбранные признаки: ['content_type', 'age', 'income', 'sex', 'kids_flg', 'total_dur', 'release_year']
Отсутствующие признаки: []

Финальные признаки для модели: ['content_type_le', 'age_le', 'income_le', 'sex_le', 'kids_flg', 'total_dur', 'release_year']


In [42]:
from sklearn.model_selection import train_test_split

# Определим X (признаки) и y (целевая переменная)
X = train_full[final_features]
y = train_full['target']

# Разделим на обучающую и валидационную выборки
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Форма X_train: {X_train.shape}")
print(f"Форма X_val: {X_val.shape}")
print(f"Распределение классов в y_train:\n{y_train.value_counts()}")
print(f"Распределение классов в y_val:\n{y_val.value_counts()}")

Форма X_train: (738225, 7)
Форма X_val: (184557, 7)
Распределение классов в y_train:
target
0    393694
1    344531
Name: count, dtype: int64
Распределение классов в y_val:
target
0    98424
1    86133
Name: count, dtype: int64


In [43]:
print(X_train.head(10).to_markdown())

|        |   content_type_le |   age_le |   income_le |   sex_le |   kids_flg |   total_dur |   release_year |
|-------:|------------------:|---------:|------------:|---------:|-----------:|------------:|---------------:|
| 792744 |                 0 |        2 |           3 |        2 |          0 |         898 |           2014 |
| 126082 |                 1 |        5 |           3 |        2 |          0 |          19 |           2023 |
|  52076 |                 0 |        3 |           2 |        1 |          0 |        5952 |           2017 |
| 827744 |                 0 |        3 |           3 |        2 |          1 |         791 |           2016 |
| 760386 |                 1 |        0 |           0 |        0 |         -1 |        8485 |           2014 |
| 468414 |                 0 |        1 |           4 |        2 |          0 |           7 |           2022 |
| 169358 |                 0 |        0 |           0 |        0 |         -1 |        5897 |           2022 |
|

In [44]:
import lightgbm as lgb
from sklearn.metrics import f1_score

# Определим категориальные признаки (их индексы в final_features)
# ['content_type_le', 'age_le', 'income_le', 'sex_le', 'kids_flg', 'total_dur', 'release_year']
# Индексы закодированных категориальных признаков: 0, 1, 2, 3
categorical_indices = [0, 1, 2, 3]

# Создадим датасеты LGBM
lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_indices)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=categorical_indices)

# Параметры модели
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',  # или 'auc', 'f1'
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,  # Убираем вывод каждые N итераций
    'seed': 42
}

In [45]:
# 1. Объединим test_df с users_df и items_df
test_full = test_df.merge(users_df, on='user_id', how='left')
test_full = test_full.merge(items_df, on='item_id', how='left')

# 2. Заполним пропуски так же, как в train_full
cat_features_to_fill = ['age', 'income', 'sex']
for col in cat_features_to_fill:
    test_full[col] = test_full[col].fillna('Unknown')

num_features_to_fill = ['for_kids', 'release_year', 'kids_flg']
for col in num_features_to_fill:
    test_full[col] = test_full[col].fillna(-1)

# 3. Применим Label Encoding, используя сохранённые объекты из le_dict
for col in categorical_features:  # ['content_type', 'age', 'income', 'sex']
    # Применим тот же LabelEncoder, что и в train
    le = le_dict[col]
    # Важно: если в тесте есть значения, которых не было в train, transform вызовет ошибку.
    # Поэтому заменим их на 'Unknown' (или любое другое, которое точно есть в train).
    # Убедимся, что все значения в test присутствуют в fit-списке
    test_full[col] = test_full[col].apply(lambda x: x if x in le.classes_ else 'Unknown')
    test_full[col + '_le'] = le.transform(test_full[col].astype(str))

# 4. Выберем финальные признаки
X_test = test_full[final_features]

print("Форма X_test после подготовки:")
print(X_test.shape)
print("Колонки X_test:")
print(list(X_test.columns))

Форма X_test после подготовки:
(100577, 7)
Колонки X_test:
['content_type_le', 'age_le', 'income_le', 'sex_le', 'kids_flg', 'total_dur', 'release_year']


In [46]:
# Список текстовых колонок для объединения
text_features = ['genres', 'countries', 'studios', 'directors', 'actors', 'keywords']

# Заменим NaN на пустую строку в текстовых колонках для корректного объединения
for col in text_features:
    train_full[col] = train_full[col].fillna('')
    test_full[col] = test_full[col].fillna('')

# Создадим объединённую текстовую колонку
# Используем пробел как разделитель между значениями разных колонок
train_full['combined_text'] = train_full[text_features].apply(lambda x: ' '.join(x), axis=1)
test_full['combined_text'] = test_full[text_features].apply(lambda x: ' '.join(x), axis=1)

print("Колонка 'combined_text' создана в train_full и test_full.")
print("\nПример содержимого 'combined_text' в train_full:")
print(train_full['combined_text'].head(3))

Колонка 'combined_text' создана в train_full и test_full.

Пример содержимого 'combined_text' в train_full:
0    Genre_000008, Genre_000035, Genre_000007 Count...
1    Genre_000005 Country_000005  Director_000357 A...
2    Genre_000035, Genre_000024, Genre_000015, Genr...
Name: combined_text, dtype: object


In [48]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# --- Выберем признаки ---
# Текстовый признак
text_feature = 'combined_text'

# Числовые признаки (можно добавить/убрать по необходимости)
numeric_features = ['total_dur', 'release_year', 'age_rating', 'kids_flg']

# Категориальные признаки (уже закодированы в ..._le)
categorical_features = ['content_type_le', 'age_le', 'income_le', 'sex_le']

# Финальный список признаков
final_features_pipeline = categorical_features + numeric_features

# --- Подготовим X и y ---
X = train_full[[text_feature] + final_features_pipeline]
y = train_full['target']

print("Форма X (до разделения):", X.shape)
print("Признаки:", list(X.columns))

Форма X (до разделения): (922782, 9)
Признаки: ['combined_text', 'content_type_le', 'age_le', 'income_le', 'sex_le', 'total_dur', 'release_year', 'age_rating', 'kids_flg']


In [49]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from lightgbm import LGBMClassifier

# --- Определяем трансформеры для разных типов признаков ---
# Для текстовой колонки используем TfidfVectorizer
text_transformer = Pipeline(steps=[
    ('tfidf', TfidfVectorizer(max_features=2000, stop_words=None, ngram_range=(1, 2)))
])

# Для числовых колонок используем StandardScaler
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Объединяем трансформеры с помощью ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_transformer, 'combined_text'), # Применить TF-IDF к 'combined_text'
        ('num', numeric_transformer, ['total_dur', 'release_year', 'age_rating', 'kids_flg']) # Применить StandardScaler к числовым
    ],
    remainder='passthrough'  # Все остальные колонки (categorical_features) будут переданы как есть
)

# --- Создаём полный Pipeline ---
# Сначала предобработка (preprocessor), затем обучение модели (LGBMClassifier)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        objective='binary',
        metric='binary_logloss',
        boosting_type='gbdt',
        num_leaves=127,
        learning_rate=0.05,
        feature_fraction= 0.8,
        bagging_fraction= 0.7,
        bagging_freq= 5,
        verbose=-1,
        seed=42,
        # n_estimators=1000 # Можно указать фиксированное количество деревьев вместо early_stopping, если он не поддерживается внутри Pipeline
    ))
])

print("Pipeline создан.")

Pipeline создан.


In [50]:
from sklearn.model_selection import train_test_split

# Разделим данные
X_train_p, X_val_p, y_train_p, y_val_p = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Форма X_train_p: {X_train_p.shape}")
print(f"Форма X_val_p: {X_val_p.shape}")

# Обучим модель (включая предобработку) на X_train_p
print("\nОбучаем модель...")
model_pipeline.fit(X_train_p, y_train_p)

# Сделаем предсказания на валидации
print("\nДелаем предсказания на валидации...")
y_val_pred_proba = model_pipeline.predict_proba(X_val_p)[:, 1]  # Вероятность класса 1
y_val_pred = (y_val_pred_proba > 0.5).astype(int)

# Посчитаем F1-меру
f1_pipeline = f1_score(y_val_p, y_val_pred, average='macro')
print(f"\nF1-macro на валидации (Pipeline): {f1_pipeline:.4f}")

Форма X_train_p: (738225, 9)
Форма X_val_p: (184557, 9)

Обучаем модель...


: 